In [ ]:
import newkernelo as lib
import numpy as np
import time
import os.path
import json
import logging
logging.getLogger().setLevel(logging.INFO)

In [ ]:
model =lib.FunctionalModel().get_D_dimension()
# help(lib.FunctionalModel)
# a = model.get_D_dimension()
print(model)

In [ ]:
model = lib.TestModel()
print(model.__doc__)
print(lib.ShkuratovModel.__doc__)
# print(lib.FunctionalModel.__doc__)
# print(lib.ExternalPythonModel.__doc__)
# print(lib.ExternalPythonModel.__init__.__doc__)
help(lib.TestModel)


## Test Model

In [ ]:
model = lib.TestModel()
e = np.exp(1)
x_test = np.zeros(4)
y_test = model.F(x_test)
true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
print("y_test")
print(y_test)
print("true_result")
print(true_result)

In [ ]:
e = np.exp(1)
x = np.ones(4)

# y = np.zeros(9)
# t0 = time.time()
# for i in range(10000000): #10000000
#     model.F(x,y)
# tf = time.time() - t0
# print("Time F by reference: {}".format(tf))

# t0 = time.time()
# for i in range(10000000):
#     z = model.F(x)
# tf = time.time() - t0
# print("Time F by value: {}".format(tf))

# true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
# print("true_result")
# print(true_result)

z = model.F(x)
print(z)
print(x)
print(x.shape)
model.to_physic(x) # does nothing
print(x)
print(x.shape)
w = model.to_physic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
model.from_physic(x)
print(x)
y = model.from_physic(x)
print(x)
print(y)
z = model.from_physic(y)
print(x)
print(y)
print(z)

## Shkuratov

In [ ]:
print(os.getcwd())
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]


# # Create JSON file with geometries
geometries = np.empty((row_size,col_size))
var_geom = ["inc", "eme", "phi"]
for j in range(3):
    i=0
    for v in data[var_geom[j]]:
        geometries[i,j] = v # geometries.shape = (D,3)
        i+=1
# print(geometries)
print(geometries.shape)

# geom = {'geometries': [[geometries[j,:].tolist()] for j in range(3)]
# }
# with open('geometries_shkuratov.json', 'w') as fp:
#     json.dump(geom, fp)

## INTEGRATION au code C++
variant = "5p"
physicalModel = lib.ShkuratovModel(geometries, variant, scalingCoeffs, offset)


### TEST
N = 100
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])



# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = physicalModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])


## Hapke

In [ ]:
with open('../../data_ref_hapke6p_geom70.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "six", 30.0,1,0)
print(hapkeModel.get_D_dimension())
print(hapkeModel.get_L_dimension())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n])
print(result)

In [ ]:
with open('../../data_ref_hapke4p_geom70.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "four", 30.0,1,0)
print(hapkeModel.get_D_dimension())
print(hapkeModel.get_L_dimension())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n,:10])
print(result[:10])

### NOTE : le expected_results[n][2] et result[2] sont différents.... mais les autres sont égaux. à voir...

In [ ]:
print(hapkeModel.get_D_dimension())
print(hapkeModel.get_L_dimension())
x = np.ones(hapkeModel.get_L_dimension()) / 10

print(x)
print(x.shape)
hapkeModel.to_physic(x) # does nothing
print(x)
print(x.shape)
w = hapkeModel.to_physic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
hapkeModel.from_physic(x)
print(x)
y = hapkeModel.from_physic(x)
print(x)
print(y)
z = hapkeModel.from_physic(y)
print(x)
print(y)
print(z)

## External model

In [ ]:
externalPythonModel = lib.ExternalPythonModel("ShkuratovModel5p", "ShkuratovModel5pPython", "../../pytest/models/")

print(externalPythonModel.get_D_dimension())
print(externalPythonModel.get_L_dimension())

In [ ]:
x = np.ones(externalPythonModel.get_L_dimension())

print(x)
print(x.shape)
print("=========== To physic ==========")
externalPythonModel.to_physic(x) # does nothing
print(x)
print(x.shape)
w = externalPythonModel.to_physic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
externalPythonModel.from_physic(x)
print(x)
y = externalPythonModel.from_physic(x)
print(x)
print(y)
z = externalPythonModel.from_physic(y)
print(x)
print(y)
print(z)

In [ ]:
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]



### TEST
N = 3
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])


# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = externalPythonModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])